# Tabular Readmission Risk Modeling & Explainability (SMOTE, XGBoost, Optuna, LightGBM & TreeSHAP)
Compares baseline `XGBoost`, `SMOTE`, class-weighted `XGBoost`, and multi-algorithm **Optuna** hyperparameter optimization (`XGBoost`, `LightGBM`, `CatBoost`, `SVM`), then saves the winning `LightGBM` booster to `lightgbm_model.txt`.

In [ ]:
# !pip install -q datasets huggingface_hub scikit-learn imbalanced-learn xgboost lightgbm catboost optuna shap joblib pandas numpy


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
from __future__ import annotations
import warnings
from typing import Any, Dict, List, Tuple
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import shap
from huggingface_hub import hf_hub_download
from imblearn.over_sampling import SMOTE
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/yash/Library/A-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load & Impute Inpatient Readmission Features

In [4]:
csv_path = hf_hub_download(
    repo_id="Bena345/diabetes-readmission",
    filename="train.csv",
    repo_type="dataset"
)

In [5]:
from pandas.api.types import is_numeric_dtype

def load_tabular_readmission_dataset(csv_file_path: str, max_rows: int = 20000) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df_raw = pd.read_csv(csv_file_path, low_memory=False)
    if len(df_raw) > max_rows:
        df_raw = df_raw.sample(n=max_rows, random_state=42).reset_index(drop=True)

    candidate_numeric = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    available_cols = [c for c in candidate_numeric if c in df_raw.columns]

    if "age" in df_raw.columns and not is_numeric_dtype(df_raw["age"]):
        df_raw["age_years"] = (
            df_raw["age"]
            .astype(str)
            .str.extract(r"(\d+)")[0]
            .astype(float)
            + 5.0
        )
        available_cols.insert(0, "age_years")
    elif "age" in df_raw.columns:
        available_cols.insert(0, "age")

    X_df = df_raw[available_cols].apply(pd.to_numeric, errors="coerce")
    target_col = "readmitted" if "readmitted" in df_raw.columns else df_raw.columns[-1]
    raw_target = df_raw[target_col].astype(str).str.strip().str.upper()
    if set(raw_target.unique()).issubset({"0", "1", "0.0", "1.0"}):
        y = raw_target.astype(float).astype(int).to_numpy()
    else:
        y = raw_target.isin(["<30", "1", "TRUE", "YES"]).astype(int).to_numpy()

    return X_df, y, available_cols

In [6]:
pd.read_csv(csv_path).columns

Index(['encounter_id', 'patient_nbr', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'number_diagnoses', 'age',
       'weight', 'race', 'gender', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [7]:
pd.read_csv(csv_path)["age"].head()

0    [80-90)
1    [70-80)
2    [60-70)
3    [70-80)
4    [40-50)
Name: age, dtype: str

In [8]:
X_df, y_tab, feature_names = load_tabular_readmission_dataset(csv_path, max_rows=20000)
imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X_df)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_imputed, y_tab, test_size=0.2, stratify=y_tab, random_state=42
)
print(f"Training shape: {X_tr.shape} | Validation shape: {X_val.shape}")

Training shape: (16000, 9) | Validation shape: (4000, 9)


In [9]:
X_df.head()

,age_years,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
0,85.0,4,63,0,5,0,0,1,9
1,75.0,2,1,3,11,0,0,0,4
2,65.0,9,42,3,19,0,0,0,9
3,75.0,4,60,0,11,1,0,6,8
4,65.0,5,26,3,11,0,0,0,9


In [10]:
X_df.head()

,age_years,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
0,85.0,4,63,0,5,0,0,1,9
1,75.0,2,1,3,11,0,0,0,4
2,65.0,9,42,3,19,0,0,0,9
3,75.0,4,60,0,11,1,0,6,8
4,65.0,5,26,3,11,0,0,0,9


In [11]:
X_tr[:5]

array([[75.,  4., 39.,  0., 11.,  0.,  0.,  0.,  9.],
       [55.,  5., 36.,  1., 14.,  0.,  0.,  0.,  9.],
       [65.,  1., 24.,  0.,  8.,  0.,  0.,  0.,  6.],
       [85.,  2., 16.,  0., 12.,  1.,  1.,  0.,  8.],
       [65.,  3., 50.,  1., 11.,  0.,  0.,  0.,  4.]])

In [12]:
y_tr[:5]

array([0, 0, 0, 0, 0])

## Baseline XGBoost & SMOTE Comparison

In [13]:
smote = SMOTE(random_state=42)
X_tr_bal, y_tr_bal = smote.fit_resample(X_tr, y_tr)

scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.1,
    reg_lambda=2,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_tr_bal, y_tr_bal)
y_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)
print(f"Weighted XGBoost -> ROC-AUC: {roc_auc_score(y_val, y_prob_xgb):.4f} | F1: {f1_score(y_val, y_pred_xgb):.4f}")

Weighted XGBoost -> ROC-AUC: 0.6026 | F1: 0.2277


In [18]:
X_tr_bal[:5]

array([[75.,  4., 39.,  0., 11.,  0.,  0.,  0.,  9.],
       [55.,  5., 36.,  1., 14.,  0.,  0.,  0.,  9.],
       [65.,  1., 24.,  0.,  8.,  0.,  0.,  0.,  6.],
       [85.,  2., 16.,  0., 12.,  1.,  1.,  0.,  8.],
       [65.,  3., 50.,  1., 11.,  0.,  0.,  0.,  4.]])

## Optuna Multi-Algorithm Hyperparameter & Threshold Optimization (`XGBoost`, `LightGBM`, `CatBoost`, `SVM`)

In [ ]:
def build_model(params: Dict[str, Any]):
    classifier = params["classifier"]
    if classifier == "xgboost":
        return xgb.XGBClassifier(
            n_estimators=params["xgb_n_estimators"],
            max_depth=params["xgb_max_depth"],
            learning_rate=params["xgb_learning_rate"],
            min_child_weight=params["xgb_min_child_weight"],
            subsample=params["xgb_subsample"],
            colsample_bytree=params["xgb_colsample_bytree"],
            scale_pos_weight=params["xgb_scale_pos_weight"],
            reg_alpha=params["xgb_reg_alpha"],
            reg_lambda=params["xgb_reg_lambda"],
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1,
        )
    elif classifier == "lightgbm":
        return lgb.LGBMClassifier(
            n_estimators=params["lgb_n_estimators"],
            max_depth=params["lgb_max_depth"],
            learning_rate=params["lgb_learning_rate"],
            num_leaves=params["lgb_num_leaves"],
            min_child_samples=params["lgb_min_child_samples"],
            subsample=params["lgb_subsample"],
            subsample_freq=1,
            colsample_bytree=params["lgb_colsample_bytree"],
            scale_pos_weight=params["lgb_scale_pos_weight"],
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )
    elif classifier == "catboost":
        return CatBoostClassifier(
            iterations=params["cat_iterations"],
            depth=params["cat_depth"],
            learning_rate=params["cat_learning_rate"],
            auto_class_weights=params["cat_auto_weights"],
            l2_leaf_reg=params["cat_l2_leaf_reg"],
            random_seed=42,
            verbose=False,
        )
    elif classifier == "svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                C=params["svm_c"],
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42,
            )),
        ])


def objective(trial: optuna.Trial) -> float:
    classifier_name = trial.suggest_categorical(
        "classifier", ["xgboost", "lightgbm", "catboost", "svm"]
    )
    if classifier_name == "xgboost":
        trial.suggest_int("xgb_n_estimators", 100, 600)
        trial.suggest_int("xgb_max_depth", 3, 8)
        trial.suggest_float("xgb_learning_rate", 0.01, 0.1, log=True)
        trial.suggest_int("xgb_min_child_weight", 5, 30)
        trial.suggest_float("xgb_subsample", 0.6, 0.95)
        trial.suggest_float("xgb_colsample_bytree", 0.6, 0.95)
        trial.suggest_float("xgb_scale_pos_weight", 2.0, 8.0)
        trial.suggest_float("xgb_reg_alpha", 1e-3, 10.0, log=True)
        trial.suggest_float("xgb_reg_lambda", 1e-3, 10.0, log=True)
    elif classifier_name == "lightgbm":
        trial.suggest_int("lgb_n_estimators", 100, 600)
        trial.suggest_int("lgb_max_depth", 3, 8)
        trial.suggest_float("lgb_learning_rate", 0.01, 0.1, log=True)
        trial.suggest_int("lgb_num_leaves", 8, 128)
        trial.suggest_int("lgb_min_child_samples", 10, 50)
        trial.suggest_float("lgb_subsample", 0.6, 0.95)
        trial.suggest_float("lgb_colsample_bytree", 0.6, 0.95)
        trial.suggest_float("lgb_scale_pos_weight", 2.0, 8.0)
    elif classifier_name == "catboost":
        trial.suggest_int("cat_iterations", 100, 500)
        trial.suggest_int("cat_depth", 3, 8)
        trial.suggest_float("cat_learning_rate", 0.01, 0.1, log=True)
        trial.suggest_categorical("cat_auto_weights", [None, "Balanced", "SqrtBalanced"])
        trial.suggest_float("cat_l2_leaf_reg", 1e-3, 10.0, log=True)
    else:
        trial.suggest_float("svm_c", 0.1, 10.0, log=True)

    clf = build_model(trial.params)
    clf.fit(X_tr_bal, y_tr_bal)
    val_probs = clf.predict_proba(X_val)[:, 1]

    best_f1 = 0.0
    best_threshold = 0.5
    for threshold in np.arange(0.05, 0.81, 0.01):
        preds = (val_probs >= threshold).astype(int)
        score = float(f1_score(y_val, preds, zero_division=0))
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)

    trial.set_user_attr("best_threshold", best_threshold)
    return best_f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25)

best_params = study.best_params
best_model = build_model(best_params)
best_model.fit(X_tr_bal, y_tr_bal)
best_threshold = study.best_trial.user_attrs["best_threshold"]

print("=== OPTUNA SEARCH COMPLETED ===")
print("Best Algorithm:", best_params["classifier"])
print("Best Parameters:", best_params)
print("Best Validation F1:", round(study.best_value, 4))
print("Best Threshold:", best_threshold)

## Save Winning LightGBM Model (`lightgbm_model.txt`) & Test TreeSHAP Explanation

In [15]:
# best_model.booster_.save_model("lightgbm_model.txt")
loaded_model = lgb.Booster(model_file="lightgbm_model.txt")
best_threshold=0.25
y_prob = loaded_model.predict(X_val)
y_pred = (y_prob >= best_threshold).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print("PR-AUC:", average_precision_score(y_val, y_prob))
print("Brier Score:", brier_score_loss(y_val, y_prob))
print("F1:", f1_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred, zero_division=0))

ROC-AUC: 0.6210556164295068
PR-AUC: 0.17843275320134214
Brier Score: 0.11198206957092201
F1: 0.2522414823670054

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.72      0.80      3546
           1       0.17      0.46      0.25       454

    accuracy                           0.69      4000
   macro avg       0.54      0.59      0.53      4000
weighted avg       0.83      0.69      0.74      4000



In [16]:
def predict_and_explain(booster: lgb.Booster, x_imp_row: np.ndarray, feat_cols: List[str]) -> Dict[str, Any]:
    explainer = shap.TreeExplainer(booster)
    prob = float(booster.predict(x_imp_row)[0])
    shap_vals = explainer.shap_values(x_imp_row)
    if isinstance(shap_vals, list):
        sv_row = shap_vals[1][0]
    elif np.asarray(shap_vals).ndim == 3:
        sv_row = shap_vals[0, :, 1]
    else:
        sv_row = shap_vals[0]

    ranked = sorted(
        zip(feat_cols, x_imp_row[0], sv_row),
        key=lambda item: abs(float(item[2])),
        reverse=True,
    )
    top_drivers = [
        {
            "feature": fname,
            "value": round(float(val), 2),
            "shap_log_odds_impact": round(float(sv), 4),
            "direction": "increases_risk" if sv > 0 else "decreases_risk",
        }
        for fname, val, sv in ranked[:5]
    ]
    return {
        "readmission_or_icu_risk_prob": round(prob, 4),
        "risk_tier": "HIGH" if prob >= 0.40 else ("MODERATE" if prob >= 0.25 else "LOW"),
        "top_treeshap_drivers": top_drivers,
    }

print(predict_and_explain(loaded_model, X_val[:1], feature_names))

{'readmission_or_icu_risk_prob': 0.1882, 'risk_tier': 'LOW', 'top_treeshap_drivers': [{'feature': 'time_in_hospital', 'value': 6.0, 'shap_log_odds_impact': 0.1941, 'direction': 'increases_risk'}, {'feature': 'number_inpatient', 'value': 0.0, 'shap_log_odds_impact': -0.1479, 'direction': 'decreases_risk'}, {'feature': 'num_procedures', 'value': 3.0, 'shap_log_odds_impact': -0.1046, 'direction': 'decreases_risk'}, {'feature': 'number_diagnoses', 'value': 5.0, 'shap_log_odds_impact': -0.0879, 'direction': 'decreases_risk'}, {'feature': 'age_years', 'value': 85.0, 'shap_log_odds_impact': 0.0363, 'direction': 'increases_risk'}]}
